In [1]:
import matplotlib.pyplot as plt
import numpy as np
from utils import ImageBatchGenerator,AnimalsDataset
from PIL import Image
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision 
import torch.nn.functional as F

c:\Users\ayhan\anaconda3\Lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: 'Could not find module 'C:\Users\ayhan\anaconda3\Lib\site-packages\torchvision\image.pyd' (or one of its dependencies). Try using the full path with constructor syntax.'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


### now lets set our baseline model and get the baseline score 

In [5]:
# define the baseline CNN model
class BaselineCNN(nn.Module):
    def __init__(self,in_channels:int=3,num_classes:int=2)->None:
        super(BaselineCNN,self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels,out_channels=6,kernel_size=(3,3),stride=(2,2),padding=(1,1))
        self.pool = nn.MaxPool2d(kernel_size=(2,2),stride=(2,2))
        self.conv2 = nn.Conv2d(in_channels=6,out_channels=12,kernel_size=(3,3),stride=(2,2),padding=(1,1))
        self.conv3 = nn.Conv2d(in_channels=12,out_channels=24,kernel_size=(3,3),stride=(2,2),padding=(1,1))
        self.fc1 = nn.LazyLinear(out_features=64) # Linear(in_features=2028, out_features=2, bias=True)
        self.fc2 = nn.LazyLinear(out_features=num_classes) 


    def forward(self,X):
        x =  F.relu(self.conv1(X))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        self.pool(x)
        x = F.relu(self.conv3(x))
        x = x.reshape(x.shape[0],-1) # flattening
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x


In [10]:
# basic check
base_model = BaselineCNN()
x= torch.randn(32,3,100,100)
base_model(x).shape, base_model

c:\Users\ayhan\anaconda3\Lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


(torch.Size([32, 2]),
 BaselineCNN(
   (conv1): Conv2d(3, 6, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
   (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
   (conv2): Conv2d(6, 12, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
   (conv3): Conv2d(12, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
   (fc1): Linear(in_features=1176, out_features=64, bias=True)
   (fc2): Linear(in_features=64, out_features=2, bias=True)
 ))

In [11]:
dogs_path = r"C:\Users\ayhan\Desktop\ml-collection\data\ten_animals\raw-img\dog"
cats_path = r"C:\Users\ayhan\Desktop\ml-collection\data\ten_animals\raw-img\cat"
batch_size = 128
batch_generator = ImageBatchGenerator({"dog": dogs_path, "cat": cats_path}, batch_size=32, network_input_size=(224, 224))
#val_tuple, test_tuple = batch_generator.return_val_test_batches()
train_dataset = AnimalsDataset(batch_generator)
val_dataset = AnimalsDataset(batch_generator, img_paths=batch_generator.val_img_paths, labels=batch_generator.val_labels)
test_dataset = AnimalsDataset(batch_generator, img_paths=batch_generator.test_img_paths, labels=batch_generator.test_labels)
# Create dataloaders
batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


n_samples 653


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# hyperparams
num_classes = 2
lr = 0.01
num_epochs = 5
in_channels = 3
n_iters = 200
base_model = BaselineCNN(in_channels=in_channels,num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss() # binary classification
optimizer = optim.AdamW(base_model.parameters(),lr=lr)
for epoch in range(num_epochs):
    for batch_idx, (batch_data,labels) in enumerate(train_dataloader):
        if batch_idx == n_iters:
            print("########################################################")
            break
        batch_data = batch_data.to(device)
        labels = labels.to(device)        
        logits = base_model(batch_data)
        L = criterion(logits,labels)
        optimizer.zero_grad()
        L.backward()
        optimizer.step()
        if batch_idx % 20 ==0:
            preds = logits.max(1).indices
            accuracy = ((preds==labels).sum().item())/batch_size
            print(f"epoch {epoch+1}/{num_epochs}: train loss: {L.item()} | accuracy: {accuracy*100}%")
    
            

epoch 1/5: train loss: 0.6393569707870483 | accuracy: 96.875%
epoch 1/5: train loss: 0.3152054250240326 | accuracy: 90.625%
epoch 1/5: train loss: 0.2396039366722107 | accuracy: 93.75%
epoch 1/5: train loss: 0.2575214207172394 | accuracy: 93.75%
epoch 1/5: train loss: 0.15696363151073456 | accuracy: 96.875%
epoch 1/5: train loss: 0.3197886645793915 | accuracy: 90.625%
epoch 1/5: train loss: 0.15354357659816742 | accuracy: 96.875%
epoch 1/5: train loss: 0.14473091065883636 | accuracy: 96.875%
epoch 1/5: train loss: 0.23409566283226013 | accuracy: 93.75%
epoch 2/5: train loss: 0.23760364949703217 | accuracy: 93.75%
epoch 2/5: train loss: 0.37678614258766174 | accuracy: 87.5%
epoch 2/5: train loss: 0.4237058758735657 | accuracy: 87.5%
epoch 2/5: train loss: 0.38631942868232727 | accuracy: 87.5%
epoch 2/5: train loss: 0.157607764005661 | accuracy: 96.875%
epoch 2/5: train loss: 0.23417620360851288 | accuracy: 93.75%
epoch 2/5: train loss: 0.43798404932022095 | accuracy: 84.375%
epoch 2/5: 

In [13]:
base_model.eval()
total_correct = 0
total_samples = 0
with torch.no_grad():
    for batch_idx, (batch_data,labels) in enumerate(val_dataloader):
        batch_data = batch_data.to(device)
        labels = labels.to(device)        
        logits = base_model(batch_data)
        L = criterion(logits,labels)
        preds = logits.max(1).indices
        total_correct += (preds==labels).sum().item()
        total_samples += len(labels)
    print(f"Validation accuracy: {total_correct/total_samples*100}%")


Validation accuracy: 50.0%
